# Module 1 : Réseau de Neurones Dense (DNN) — Classification MNIST 🧠

**Fondations Deep Learning | Papa Séga WADE — [papasegawade.com](https://papasegawade.com)**

---

## 🎯 Objectifs de ce module

À la fin de ce module, vous serez capable de :
- Comprendre ce qu'est un **neurone artificiel** et une **couche Dense**
- Construire un DNN avec **TensorFlow/Keras**
- Entraîner et évaluer un modèle sur le dataset **MNIST** (~98% accuracy)
- Interpréter les **courbes d'apprentissage**

---

## 📐 Rappel : Qu'est-ce qu'un neurone artificiel ?

Un neurone artificiel calcule une **somme pondérée** de ses entrées, puis applique une **fonction d'activation** :

$$z = w_1 x_1 + w_2 x_2 + ... + w_n x_n + b$$
$$a = f(z)$$

Où :
- $x_i$ : les entrées
- $w_i$ : les **poids** (appris pendant l'entraînement)
- $b$ : le **biais**
- $f$ : la **fonction d'activation** (ReLU, Softmax...)

Un **DNN (Dense Neural Network)** empile plusieurs couches de neurones.

## 1. Imports et chargement des données

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)

# Chargement du dataset MNIST (inclus dans Keras)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Labels (classes) : {np.unique(y_train)}")  # 0 à 9

## 2. Exploration visuelle du dataset

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(20, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis('off')
plt.suptitle("20 premiers exemples du dataset MNIST", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nValeurs de pixel : min={X_train.min()} max={X_train.max()}")
print(f"Forme d'une image : {X_train[0].shape} => 28 lignes x 28 colonnes de pixels")

## 3. Prétraitement

Un DNN attend un **vecteur 1D** en entrée, pas une image 2D. On doit donc :
1. **Aplatir** (Flatten) les images 28×28 → vecteur de 784 valeurs
2. **Normaliser** les pixels de [0, 255] → [0, 1]

In [ ]:
# Normalisation (0-255 → 0-1)
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

# Flatten : 28x28 → 784
X_train_flat = X_train.reshape(-1, 28*28)
X_test_flat  = X_test.reshape(-1, 28*28)

print(f"Shape après flatten : {X_train_flat.shape}")

# One-hot encoding des labels : 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat  = keras.utils.to_categorical(y_test, 10)

print(f"Label 3 encodé : {y_train_cat[np.where(y_train==3)[0][0]]}")

## 4. Architecture du DNN

```
Input (784)
    ↓
Dense (512 neurones) + ReLU
    ↓
Dropout (0.3)
    ↓
Dense (256 neurones) + ReLU
    ↓
Dropout (0.3)
    ↓
Dense (10 neurones) + Softmax  ← Prédiction des 10 classes
```

In [ ]:
model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=(784,), name='couche_1'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu', name='couche_2'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax', name='sortie')
])

model.summary()

## 5. Compilation et entraînement

| Paramètre | Valeur | Pourquoi |
|-----------|--------|----------|
| `optimizer` | `adam` | Adaptatif, fonctionne bien en pratique |
| `loss` | `categorical_crossentropy` | Classification multi-classes |
| `metrics` | `accuracy` | Facile à interpréter |

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_flat, y_train_cat,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## 6. Résultats et interprétation

In [ ]:
# Courbes d'apprentissage
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Précision (Accuracy)')
ax1.set_xlabel('Epochs')
ax1.legend()

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Perte (Loss)')
ax2.set_xlabel('Epochs')
ax2.legend()

plt.suptitle('Courbes d\'apprentissage — DNN MNIST', fontsize=14)
plt.tight_layout()
plt.show()

test_loss, test_acc = model.evaluate(X_test_flat, y_test_cat, verbose=0)
print(f"\n🏆 Test Accuracy : {test_acc*100:.2f}%")

In [ ]:
# Visualisation de quelques prédictions
preds = model.predict(X_test_flat[:12])
pred_classes = np.argmax(preds, axis=1)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i], cmap='gray')
    color = 'green' if pred_classes[i] == y_test[i] else 'red'
    ax.set_title(f"Vrai: {y_test[i]} | Pred: {pred_classes[i]}", color=color)
    ax.axis('off')
plt.suptitle('Prédictions du modèle DNN', fontsize=13)
plt.tight_layout()
plt.show()

---

## ✅ Bilan du Module 1

| Concept | Ce que vous avez appris |
|---------|------------------------|
| Neurone artificiel | Somme pondérée + Activation |
| DNN | Empilement de couches Dense |
| Overfitting | Écart entre courbes train/val → Dropout |
| Softmax | Convertit les sorties en probabilités |

➡️ **Module suivant :** Le CNN — exploiter la structure spatiale des images.